# Thí nghiệm <model> / <method> / <expNNN>

Notebook này chạy một thí nghiệm rồi ghi kết quả. **Bấm Run all** là đủ.

Nó tự làm phần khó TRƯỚC khi chạy: kéo ĐÚNG bản code đã ghim ở cell đầu (nên chạy trên Colab hay trên máy cá nhân đều ra cùng kết quả), rồi kiểm dữ liệu, thiết bị và quyền ghi. Có gì chưa đúng thì nó **dừng và in ra danh sách việc phải sửa**, thay vì chạy nửa chừng rồi hỏng.

Cell đầu do `python scripts/pin.py` ghi - sửa tay sẽ bị ghi đè ở lần ghim sau. Các cell còn lại là bản mẫu trong `templates/experiment/notebook.ipynb`.

In [ ]:
# --- BẢN CODE ĐÃ GHIM (do scripts/pin.py ghi; sửa tay sẽ bị ghi đè) ---
REPO_URL = 'https://github.com/TrieuKhac-dev/SentimentX'
REPO_BRANCH = 'experiment'
REPO_SHA = 'd9a705f65b4116e627ae74096aaf52cd41d76676'
EXP_DIR = 'qwen3-4b-instruct-2507/prompt-cot/exp001'


In [ ]:
# Chuẩn bị môi trường: đưa thư mục code về ĐÚNG commit đã ghim rồi mới `import src`.
# Ô này làm HẾT việc chuẩn bị trên Colab: kéo đúng commit, mount Drive, đặt gốc dữ liệu/kết quả, cài gói còn thiếu. Người chạy chỉ bấm Run all.
#
# VÌ SAO Ô NÀY DÙNG GIT THUẦN: trên máy mới (Colab) `src/` CHƯA tồn tại, nên không thể gọi
# `src/repo.prepare()` để kéo chính nó. Ba lệnh dưới đây đúng bằng phương án thứ hai trong
# `src/repo.plans()` (test khoá lại điều đó); sau khi có mã nguồn, `repo.prepare()` mới là nơi KIỂM
# LẠI toàn bộ (sha, nhánh cho phép, cảnh báo) - nên luật "kéo code" vẫn chỉ có một bản ở thư viện.
#
# KHÔNG dùng `git clone` trần: bản mặc định của repo có thể KHÔNG chứa `src/` (mã nguồn nằm ở nhánh
# `experiment`), clone trần xong vẫn không import được gì.
import importlib.util
import os
import pathlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules


def repo_root():
    """Gốc repo: nơi có `src/paths.py`.

    Trên Colab là thư mục code sẽ kéo về; trên máy cá nhân thì tìm từ thư mục đang đứng đi lên, để
    notebook chạy được dù Jupyter mở ở đâu.
    """
    if IN_COLAB:
        return pathlib.Path("/content/SentimentX")
    here = pathlib.Path.cwd()
    for candidate in [here] + list(here.parents):
        if (candidate / "src" / "paths.py").is_file():
            return candidate
    return here


def git(*args):
    """Chạy một lệnh git và in ra mã thoát (không giấu lỗi kéo code)."""
    done = subprocess.run(["git"] + [str(item) for item in args],
                          capture_output=True, text=True)
    print("  git {} -> {}".format(" ".join(str(item) for item in args), done.returncode))
    if done.returncode != 0:
        print("    " + ((done.stdout or "") + (done.stderr or "")).strip()[-400:])
    return done.returncode


CODE_DIR = repo_root()


def is_repo_here():
    """Thư mục code đã là git repo chưa (để khỏi clone đè lên repo có sẵn)."""
    done = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--is-inside-work-tree"],
                          capture_output=True, text=True)
    return done.returncode == 0 and done.stdout.strip() == "true"


if not (CODE_DIR / "src" / "paths.py").is_file():
    print("Chưa có mã nguồn ở {} - đang kéo commit đã ghim...".format(CODE_DIR))
    # Chạy lại ô này là chuyện thường: Runtime -> Restart session, hoặc chạy lại sau khi sửa một lỗi
    # khác. Lúc đó thư mục đã tồn tại, mà kéo code vào thư mục KHÔNG RỖNG thì thất bại với mã thoát
    # 128 (destination path already exists and is not an empty directory). Dòng lỗi đỏ đó làm người
    # đọc tưởng hỏng, trong khi chỉ cần bỏ qua bước kéo và đi thẳng tới fetch + checkout.
    if CODE_DIR.is_dir() and any(CODE_DIR.iterdir()) and not is_repo_here():
        raise SystemExit(
            "DỪNG: {} đã có sẵn nhưng KHÔNG phải git repo - nhiều khả năng là thư mục còn sót lại "
            "từ lần chạy trước hỏng giữa chừng.\nXoá nó rồi chạy lại ô này:\n  rm -rf {}"
            .format(CODE_DIR, CODE_DIR))
    if is_repo_here():
        print("  thư mục đã là git repo - bỏ qua bước kéo mới, chỉ đi tới commit đã ghim")
    else:
        CODE_DIR.mkdir(parents=True, exist_ok=True)
        git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(CODE_DIR))
    git("-C", str(CODE_DIR), "fetch", "--depth", "1", "origin", REPO_SHA)
    git("-C", str(CODE_DIR), "checkout", "--detach", REPO_SHA)
if not (CODE_DIR / "src" / "paths.py").is_file():
    # Không kéo được: in đủ thứ cần để đoán, thay vì chỉ báo thiếu file.
    print("KHÔNG thấy {}/src/paths.py".format(CODE_DIR))
    print("  thư mục đang đứng : {}".format(pathlib.Path.cwd()))
    print("  CODE_DIR tồn tại  : {}".format(CODE_DIR.is_dir()))
    if CODE_DIR.is_dir():
        names = sorted(item.name for item in CODE_DIR.iterdir())
        print("  có trong đó       : {}".format(", ".join(names)[:400] or "(rỗng)"))
        root = subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--show-toplevel"],
                              capture_output=True, text=True).stdout.strip()
        print("  là repo git       : {}".format(root or "không"))
    raise SystemExit(
        "DỪNG: không kéo được commit {} từ {}. Xem các dòng trên, hoặc Runtime -> Restart session "
        "rồi chạy lại.".format(REPO_SHA[:8], REPO_URL))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# XOÁ BỘ NHỚ ĐỆM IMPORT CŨ của kernel rồi mới import.
#
# Vì sao cần: lần chạy ĐẦU TIÊN (khi máy chưa có mã nguồn) Python đã hỏi "ở thư mục này có gói nào
# không?" và ghi nhớ câu trả lời "không" vào `sys.path_importer_cache`. Sau khi ô này kéo mã nguồn
# về, câu trả lời cũ VẪN CÒN trong bộ nhớ KERNEL (không phải bộ nhớ của ô), nên `import src` vẫn báo
# `ModuleNotFoundError: No module named 'src'` dù file đã nằm trên đĩa - và chạy lại ô cũng không
# đổi. Đây đúng là lỗi đã gặp trên Colab.
importlib.invalidate_caches()
sys.path_importer_cache.pop(str(CODE_DIR), None)

from src import paths, repo, runtime

# Nạp biến môi trường TRƯỚC khi dùng: token DagsHub và gốc dữ liệu/kết quả nằm ở file `.env`
# (máy cá nhân) hoặc Colab Secrets + `<Drive>/env/.env.colab`. Không nạp thì token có trong máy mà
# phần ghi nhận vẫn báo "thiếu token" - một lỗi im lặng rất khó đoán.
# TRÊN COLAB: mount Drive, rồi tự đặt gốc dữ liệu và gốc kết quả vào thư mục của nhóm trong Drive.
#
# Vì sao việc này nằm ở ĐÂY: người chạy notebook chỉ nên phải làm hai việc - đưa thư mục của nhóm
# (và file notebook này) lên Drive, rồi bấm Run all. Mount là việc Google bắt bấm cho phép nên không
# thể tự động hoàn toàn; mọi việc còn lại thì máy làm được: tìm thư mục nhóm bằng file đánh dấu
# `.sentimentx_root` (không cần biết tên thư mục), đặt hai gốc đường dẫn, cài gói còn thiếu.
if IN_COLAB and not pathlib.Path("/content/drive").is_mount():
    try:
        from google.colab import drive as _drive
        print("Chưa mount Drive - đang mount (Colab hỏi quyền, bấm Allow)...")
        _drive.mount("/content/drive")
    except Exception as exc:  # noqa: BLE001 - thiếu quyền/mạng thì báo, không làm chết notebook
        print("  không mount được Drive ({}).".format(exc))

drive = runtime.drive_dir() if IN_COLAB else None
# Thứ tự: `env/.env.colab` (nếu có) nạp trước, rồi hai gốc suy từ thư mục Drive tìm được - nên một
# lượt chạy bình thường KHÔNG cần tạo file env nào.
env = runtime.load_env(colab_env_file=(runtime.drive_env_file() if drive else None))
if drive:
    os.environ.setdefault("SENTIMENTX_DATA_ROOT", str(drive / "data"))
    os.environ.setdefault("SENTIMENTX_RESULTS_ROOT", str(drive / "experiments"))

print("Nơi chạy    :", runtime.env_name())
if IN_COLAB:
    if drive:
        print("Drive       : {} (nhận ra bằng file đánh dấu)".format(drive))
    else:
        print("Drive       : CHƯA thấy - dữ liệu và kết quả sẽ nằm trong máy ảo và MẤT khi hết phiên.")
        print("              Cách sửa: đưa thư mục của nhóm (có file .sentimentx_root) lên Drive rồi"
              " chạy lại ô này.")
print("Gốc dữ liệu :", paths.data_root())
print("Gốc kết quả :", paths.results_root())
print("Biến bắt buộc: có {} | thiếu {}".format(
    ", ".join(env["found"]) or "không có", ", ".join(env["missing"]) or "không thiếu"))

# Cài gói mà lần chạy cần nhưng máy ảo CHƯA có (Colab hay thiếu `bitsandbytes` cho lượng hoá 4-bit).
# Chỉ làm trên Colab, và chỉ khi thật sự thiếu: máy cá nhân tự cài theo requirements.txt.
if IN_COLAB:
    _missing = [name for name in ("transformers", "accelerate", "bitsandbytes")
                if importlib.util.find_spec(name) is None]
    if _missing:
        print("Thiếu gói {} - đang cài...".format(", ".join(_missing)))
        _installed = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing,
                                    capture_output=True, text=True)
        print("  pip install ->", _installed.returncode)
        if _installed.returncode != 0:
            print("  " + ((_installed.stdout or "") + (_installed.stderr or "")).strip()[-400:])

# Kiểm lại bằng thư viện: lệch sha là DỪNG, chứ không chạy trên bản code không rõ là bản nào.
code = repo.prepare(REPO_URL, REPO_SHA, branch=REPO_BRANCH, dest=CODE_DIR, require_branch=True)
print("Code        : {} | {} | {}".format(code["action"], code["dir"], REPO_SHA[:8]))
if not IN_COLAB and code["action"] != "dùng bản code đang có":
    print("  cảnh báo: thư mục code vừa được đưa về commit đã ghim (checkout tách rời). Máy cá nhân"
          " nên chạy notebook khi không có việc đang làm dở trong repo này.")
for warning in code["warnings"]:
    print("  cảnh báo:", warning)


In [ ]:
# Cấu hình ĐANG DÙNG: in ra để người đọc bảng điểm sau này biết nó thuộc cấu hình nào.
from src import dataset, experiments, model_config, versioning

parts = [part for part in EXP_DIR.replace("\\", "/").split("/") if part]
model_id, method, exp_id = parts
result = experiments.load(model_id, method, exp_id)
config = result["config"]
ds = dataset.load_config(config["data"]["dataset"])
version_id = versioning.compute_id(ds)

print("Thí nghiệm  : {}/{}/{}".format(model_id, method, exp_id))
print("Dataset     : {} {} -> {}".format(ds["name"], ds.get("version"), version_id))
print("Vai         : {}".format(config["data"]["roles"]))
print("Prompt      : {}".format(config["prompt"]))
print("Bài toán    : label_space={}, neutral_policy={}, not_mentioned={}".format(
    config.get("label_space"), config.get("neutral_policy"), config.get("not_mentioned")))
print("Chấm điểm   : {} mẫu/bài (null = cả split), chỉ số {}".format(
    config.get("n"), config.get("scores")))
print("Ghi nhận    : tracker={}, experiment={}".format(
    config.get("tracker"), config.get("experiment")))
print("Model       : ngưỡng cắt {} token".format(model_config.max_length(model_id)[0]))
print("              nạp {}".format(model_config.inference(model_id)))
print("\nLớp đã hợp nhất (theo thứ tự):")
for label, path in result["layers"]:
    print("  {:<12} {}".format(label, path))
if result["overrides"]:
    print("Khoá bị lớp sau đè lên:")
    for row in result["overrides"]:
        print("  ", row[0])


In [ ]:
# KIỂM TRƯỚC khi nạp model. Một lượt val tốn hàng chục phút, nên mọi thứ phải đúng từ đầu.
from src import preflight

out_dir = paths.results_dir(model_id, method, exp_id, version_id)
report = preflight.run(result, ds=ds, version_id=version_id, model_id=model_id, out_dir=out_dir)
preflight.print_report(report)
print("\nThư mục kết quả:", out_dir)
print("Trạng thái     :", report["info"].get("mode"), "-", report["info"].get("mode_reason"))

if report["problems"]:
    raise SystemExit(
        "DỪNG: còn {} việc phải sửa (xem danh sách ở trên). Sửa xong thì chạy lại ô này.".format(
            len(report["problems"])))


In [ ]:
# CHẠY thí nghiệm rồi chấm điểm: gọi THƯ VIỆN của repo, không gọi script dòng lệnh và không chép
# lại logic. `src/experiment_run.py` là nơi DUY NHẤT biết cách chạy; cửa vào dòng lệnh (dùng khi
# muốn chạy nhanh ngoài notebook) cũng gọi đúng hai hàm dưới đây nên hai đường không thể lệch nhau.
from src import experiment_run, resume

plan = experiment_run.plan(result)      # KHÔNG cần GPU: thiếu file hay sai config là dừng ngay
print("Chế độ chạy:", plan["mode"], "-", plan["reason"])
if plan["mode"] == resume.MODE_STOP:
    raise SystemExit("DỪNG: " + plan["reason"])

# Chạy tiếp được: máy đứt giữa chừng thì chạy lại ô này, phần đã xong nằm trong
# `predictions/part_*.jsonl` và điểm số vẫn tính trên CẢ split.
run_result = experiment_run.run(plan)
print("\nChế độ:", run_result["mode"], "| thư mục:", run_result["out_dir"])


## Kết quả nằm ở đâu

Mỗi lần chạy một thư mục: `experiments/<model>/<method>/<expNNN>/results/<mã dữ liệu>/<hậu tố>`.

| File | Nội dung |
| --- | --- |
| `run.log` | từng bước đã chạy, kèm lí do khi dừng; tìm `[RUN] mode=RESUME` khi chạy tiếp |
| `run_meta.json` | bản ghi lần chạy: code, config, dữ liệu, các attempt |
| `metrics.json` | chỉ số, kèm cách chấm (`label_space`, `neutral_policy`, số ô neutral bị loại) |
| `metrics.csv` | bảng dài `aspect, sentiment, metric, value` để so với các lần chạy khác |
| `mispredictions.csv` | chỉ các ô đoán sai |
| `predictions/part_*.jsonl` | kết quả ghi dần; chạy lại thì đi tiếp từ đây |
| `errors.json` | CHỈ có khi lỗi: kiểu lỗi, vết gọi, thứ còn thiếu |

Trên DagsHub: mở địa chỉ **có đuôi `.mlflow`** (in ở ô cuối). Trang repo của DagsHub chỉ hiện dữ
liệu của DagsHub nên nhìn như rỗng - đó là hai trang khác nhau.

In [ ]:
# KẾT THÚC: nhắc lại chỗ cần xem, và việc cần làm trước khi giao.
from src import tracking, utils

print("Kết quả :", utils.rel(run_result["out_dir"]))
print("DagsHub :", tracking.base.dagshub_config().get("mlflow_uri"))
print("\nĐọc kết quả theo thứ tự: metrics.json (số chính) -> metrics.csv (so với lần chạy khác)")
print("-> run.log (đã chạy những gì) -> errors.json (nếu có).")
print("\nLưu ý: ghi lại file notebook này vào git (chỉ một file) rồi đẩy lên nhánh {}.\n"
      "Sau khi ghim thì KHÔNG sửa thư mục thí nghiệm nữa cho tới khi người nhận chạy xong."
      .format(REPO_BRANCH))
